In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"], "figure.dpi": 110})

SLIM_DIR = "/Volumes/Lexar/mglurs-brain-sc-datasets/derived/regions"
FIG_DIR = "figures/allen_wmb"
RES_DIR = "results/allen_wmb"
NAMES = {'ACA': 'Anterior cingulate area', 'AI': 'Agranular insular area', 'AUD': 'Auditory areas', 'AUD-TEa-PERI-ECT': 'Auditory/temporal/perirhinal/ectorhinal areas', 'CB': 'Cerebellum', 'CTXsp': 'Cortical subplate', 'ENT': 'Entorhinal area', 'HIP': 'Hippocampal region', 'HY': 'Hypothalamus', 'LSX': 'Lateral septal complex', 'MB': 'Midbrain', 'MY': 'Medulla', 'OLF': 'Olfactory areas', 'P': 'Pons', 'PAL': 'Pallidum', 'RHP': 'Retrohippocampal region', 'RSP': 'Retrosplenial area', 'SSp': 'Primary somatosensory area', 'STRd': 'Striatum dorsal region', 'STRv': 'Striatum ventral region', 'TH': 'Thalamus', 'VIS': 'Visual areas', 'sAMY': 'Striatum-like amygdalar nuclei', 'MOp': 'Primary motor area', 'VIS-PTLp': 'Visual/posterior parietal  areas', 'TEa-PERI-ECT': 'Temporal/perirhinal/ectorhinal areas ', 'PL-ILA-ORB': 'Prelimbic/infralimbic/orbital areas ', 'SS-GU-VISC': 'Somatosensory/gustatory/visceral areas', 'MO-FRP': 'Somatomotor - Frontal pole'}
MIN_CELLS = 200
GRMS = [f"Grm{i}" for i in range(1, 9)]

_t20 = plt.get_cmap("tab20").colors
_all = [_t20[i] for i in range(0, 20, 2)] + [_t20[i] for i in range(1, 20, 2)] + list(plt.get_cmap("tab20b").colors)
PAL = [c for c in _all if not (abs(c[0] - c[1]) < 0.03 and abs(c[1] - c[2]) < 0.03)]

def class_num(c):
    n = pd.to_numeric(str(c)[:2], errors="coerce")
    return int(n) if pd.notna(n) else 0

def class_color(c):
    n = class_num(c)
    return PAL[(n - 1) % len(PAL)] if n > 0 else (0.5, 0.5, 0.5)

def is_neuron_class(c):
    return not (30 <= class_num(c) <= 34)

def neuron_type(c):
    c = str(c)
    if not is_neuron_class(c):
        return "non-neuronal"
    if "GABA" in c:
        return "GABAergic"
    if "Glut" in c:
        return "glutamatergic"
    return "other neuronal"

FOCUS_ROIS = ["PL-ILA-ORB", "ACA", "HIP", "CTXsp", "sAMY", "STRd", "STRv"]
PAIRS = [("Grm7", "Grm8"), ("Grm7", "Grm2"), ("Grm7", "Grm3"), ("Grm2", "Grm4")]
GROUP_GENES = {"Grm": ["Grm1", "Grm2", "Grm3", "Grm4", "Grm5", "Grm6", "Grm7", "Grm8"],
               "housekeeping": ["Gapdh", "Actb", "Eef1a1", "Tubb3", "Tuba1a", "Rpl13a", "B2m"],
               "other GPCRs": ["Gabbr1", "Gabbr2", "Gpr158", "Gprc5b", "Drd1", "Drd2", "Htr2a", "Chrm1", "Adora1", "Cnr1"],
               "long neuronal": ["Csmd1", "Nrxn1", "Nrxn3", "Cntnap2", "Ctnna2", "Ptprd", "Dlg2", "Auts2"]}

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

ROI = "HIP"
CHEMS = None

from scipy.stats import rankdata
import pyarrow.parquet as pq

def rcorr(a, b):
    if np.ptp(a) == 0 or np.ptp(b) == 0:
        return np.nan
    return float(np.corrcoef(rankdata(a), rankdata(b))[0, 1])

f = f"{SLIM_DIR}/{ROI}_slim.parquet"
names = pq.ParquetFile(f).schema.names
genes = [g for g in GRMS if g in names]
d = pd.read_parquet(f, columns=["class", "subclass", "n_genes_detected"] + genes)
d = d[[is_neuron_class(c) for c in d["class"]]]
rname = f"{NAMES.get(ROI, ROI)} ({ROI})"
means = d.groupby("subclass")[genes].mean()
ncell = d.groupby("subclass").size()
keep = ncell[ncell >= MIN_CELLS].index
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
rows = []
for ax, (A, B) in zip(axes.ravel(), PAIRS):
    if A not in genes or B not in genes:
        ax.axis("off")
        continue
    ta, tb = d[A].quantile(0.75), d[B].quantile(0.75)
    qa, qb = means.loc[keep, A].quantile(0.75), means.loc[keep, B].quantile(0.75)
    res = []
    for s_ in keep:
        x = d[d["subclass"] == s_]
        ha, hb = (x[A] > ta).to_numpy(), (x[B] > tb).to_numpy()
        obs, exp = float((ha & hb).mean()), float(ha.mean() * hb.mean())
        ra, rb, rz = rcorr(x[A].to_numpy(), x[B].to_numpy()), None, None
        z = x["n_genes_detected"].to_numpy()
        rab, raz, rbz = rcorr(x[A].to_numpy(), x[B].to_numpy()), rcorr(x[A].to_numpy(), z), rcorr(x[B].to_numpy(), z)
        den = np.sqrt((1 - raz ** 2) * (1 - rbz ** 2)) if np.isfinite(raz) and np.isfinite(rbz) else np.nan
        partial = (rab - raz * rbz) / den if den and np.isfinite(den) and den > 0 else np.nan
        res.append((ROI, A, B, s_, x["class"].iloc[0], len(x), float(ha.mean()), float(hb.mean()), obs, exp, obs / exp if exp > 0 else np.nan, partial,
                    bool(means.loc[s_, A] >= qa and means.loc[s_, B] >= qb)))
    r = pd.DataFrame(res, columns=["roi", "gene_a", "gene_b", "subclass", "class", "n_cells", "frac_high_a", "frac_high_b", "frac_both_high", "frac_both_expected", "obs_over_expected", "partial_spearman_depth_adj", "double_high_subclass"])
    rows.append(r)
    top = r[r["double_high_subclass"]].sort_values("frac_both_high", ascending=False).head(10)[::-1]
    if len(top) == 0:
        ax.text(0.5, 0.5, "no double-high subclasses", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(f"{A} & {B}", fontsize=11)
        continue
    y = np.arange(len(top))
    ax.barh(y + 0.19, top["frac_both_high"], height=0.36, color=plt.get_cmap("plasma")(0.2), label="observed (both high)")
    ax.barh(y - 0.19, top["frac_both_expected"], height=0.36, color=plt.get_cmap("plasma")(0.8), label="expected if independent")
    ax.set_yticks(y)
    ax.set_yticklabels(top["subclass"], fontsize=8)
    xmax = max(top["frac_both_high"].max(), top["frac_both_expected"].max())
    for yi, (_, t) in zip(y, top.iterrows()):
        ax.text(xmax * 1.02, yi, f"x{t['obs_over_expected']:.1f}, rho {t['partial_spearman_depth_adj']:.2f}", fontsize=8, va="center")
    ax.set_xlim(0, xmax * 1.45)
    ax.set_xlabel(f"fraction of cells high in both (top quartile of {NAMES.get(ROI, ROI)} neurons)", fontsize=9)
    ax.set_title(f"{A} & {B}: double-high subclasses", fontsize=11)
    ax.legend(fontsize=8, frameon=False, loc="lower right")
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
fig.suptitle(f"{rname}: do the same cells carry both receptors? Single-cell co-expression in the double-high subclasses\nx = observed / expected co-high fraction; rho = Spearman between the two receptors within the subclass, adjusted for sequencing depth (genes detected)", fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.94])
os.makedirs(f"{RES_DIR}/coexpression", exist_ok=True)
pd.concat(rows).to_csv(f"{RES_DIR}/coexpression/coexpr_cells_{ROI}.csv", index=False)
fig.savefig(f"{FIG_DIR}/{ROI}_fig9_coexpr_cells.png", dpi=170, bbox_inches="tight")
plt.show()